# Reproducing Gillespie et al. (2021) from dandiset 000115

Paper. Gillespie AK, Astudillo Maya DA, Denovellis EL, Liu DF, Kastner DB, Coulter ME,
Roumis DK, Eden UT, Frank LM (2021). *Neuron* 109, 3149-3163.
[doi:10.1016/j.neuron.2021.07.029](https://doi.org/10.1016/j.neuron.2021.07.029). The claims file
was written from the accepted author manuscript in PubMed Central (PMC8497431) with its
Supplemental Information, and from the paper's code (LorenFrankLab/Gillespie_Neuron_2021, commit
1be6a6a).

Dandiset. [DANDI:000115](https://dandiarchive.org/dandiset/000115/0.210914.1732), version
0.210914.1732 (doi:10.48324/dandi.000115/0.210914.1732). The paper cites no version, and its
code's README links the draft. 0.210914.1732 is the only published version, and it is pinned.
The draft (modified 2024-02-22) has the same 57 paths, but 55 of its files were rewritten in
January 2022 (NWB 2.4.0). The data check below compares one of them: the DIO timestamps agree to
within a microsecond, but the draft's position timestamps are no longer on the DIO clock, while
the published version's are. The published version therefore fits the analysis better as well as
being fixed.

What the dandiset holds. One NWB file per recording day for each of four rats (jaq,
roquefort, despereaux, montague): the raw 30 kHz broadband of 128 channels, port beam breaks,
lights and pumps, head position, and the StateScript log of each epoch, which records every port
entry, reward, timeout and the current goal arm. It holds no spike times, LFP, ripple times,
linearised position or decoder output. Every replay and ripple claim would have to start from
the raw broadband, about 9.1 TB in all.

Claims. 33 claims. Two can be tested here, both on behaviour. The other 31 need ripples or the
clusterless decoder run on the raw data.

| claim | tested here |
|---|---|
| fig1d-sessions-blocks | tested |
| fig1ef-reward-rate | tested |
| figs1c-swr-reward-consumption | blocked, too_large (ripple detection on raw broadband) |
| figs3abc-decode-accuracy, figs3df-goal-arm-no-remapping, figs3g-continuous-fraction, likelihood-decode-same-segment | blocked, too_large (clusterless decoding) |
| fig2f-remote-fraction, fig3a-box-majority, fig3b-search-prevgoal-enriched, fig3b-search-past-depleted, fig3b-search-future-inconsistent, fig3b-search-other, fig3c-repeat-prevgoal | blocked, too_large |
| fig3d-glm-search, fig3e-glm-repeat, fig4ab-current-goal-late, fig4c-current-goal-reward-correlation | blocked, too_large |
| fig5a-future-replay-rate, fig5b-future-correct-error, fig5c-current-goal-correct-error, figs7-early-late-correct-error, figs7e-remote-rate-error, fig5d-glm-predict-chance | blocked, too_large |
| pg-replay-visits-correlation, fig5e-pg-replay-no-avoidance, fig6a-pg-replay-persists, fig6b-visits-decay, fig6c-pg-replay-two-blocks, fig7-recency, fig8-outer-prevgoal, figs8-outbound-final-events, robustness-controls | blocked, too_large |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import replaytask as rt

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = rt.list_assets()
days = rt.load_all(assets)
print(f"loaded {len(days)} files of dandiset {rt.DANDISET_ID} version {rt.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
files = pd.DataFrame([{"subject": d["subject"], "day": d["day"], "session_id": d["session_id"],
                       "date": d["session_start"][:10], "epochs": " ".join(d["tags"]),
                       "run_epochs": len(d["runs"]), "GB": round(d["bytes"] / 1e9, 1),
                       "channels": d["n_channels"], "samples": d["n_samples"],
                       "electrode_location": ",".join(d["electrode_locations"])} for d in days])
files = files.sort_values(["subject", "day"]).reset_index(drop=True)
print("Files, run epochs and size per subject:")
print(files.groupby("subject").agg(files=("day", "size"), run_epochs=("run_epochs", "sum"),
                                   TB=("GB", lambda x: round(x.sum() / 1000, 2))).loc[rt.SUBJECTS].to_string())
print("Total size (TB):", round(files.GB.sum() / 1000, 2), "; assets:", len(assets))
print("\nTask names of the run epochs:", sorted({v[0] + " / " + v[1] for d in days for v in d["tasks"].values() if v[0] != "Sleep"}))
print("Electrode 'location' values over all files:", sorted(set(files.electrode_location)))
print("Channels per file:", sorted(set(files.channels)))
print("\n", files.to_string())

The paper used 24, 32, 23 and 33 run sessions after excluding two of subject 1 (26, 32, 23 and
33 before). The files hold 27, 32, 24 and 33 run epochs, all with the task
"Eight arm flexible spatial task / goal_delay"; jaq and despereaux have one more than the paper
reports. The electrode table gives no location for any channel ("None"), so the CA1 cell-layer
tetrodes and the corpus callosum reference tetrode the Methods name cannot be identified from the
files.

In [ ]:
runs = []
for d in days:
    for r in d["runs"]:
        tr = r["trials"]
        v = tr[~tr.lockout]
        clk = rt.statescript_to_nwb_clock(r)
        runs.append({"subject": d["subject"], "day": d["day"], "epoch": r["epoch_index"], "tag": r["epoch_tag"],
                     "minutes": round(r["duration_min"], 1), "trials": len(tr), "lockout": int(tr.lockout.sum()),
                     "valid": len(v), "rewards_log": int(v.rewarded.sum()), "reward_lines": r["n_outer_reward_lines"],
                     "pumps_dio": len(r["dio_pump_t"]), "arms_visited": v.outer.nunique(),
                     "forageNum": r["forageNum"], "outerReps_setting": r["outerReps_setting"],
                     "outerreps_printed": r["outerreps_printed"],
                     "clock_slope": None if clk is None else clk[0],
                     "clock_median_ms": None if clk is None else round(clk[2], 1),
                     "clock_max_ms": None if clk is None else round(clk[3], 1),
                     "unmatched_pumps": None if clk is None else clk[5],
                     "pos_fs": round(r.get("pos_fs", np.nan), 2), "pos_nan_t": r.get("pos_nan_t"),
                     "pos_nonincreasing": r.get("pos_nonincreasing"), "pos_unit": r.get("pos_unit"),
                     "pos_x_range_m": round(float(np.ptp(r["pos_xy"][:, 0])), 3),
                     "pos_y_range_m": round(float(np.ptp(r["pos_xy"][:, 1])), 3),
                     "sc_hash": r["sc_hash"], "run": r})
runs = pd.DataFrame(runs).sort_values(["subject", "day", "epoch"]).reset_index(drop=True)
runs["session"] = runs.groupby("subject").cumcount() + 1
show = runs.drop(columns=["run", "sc_hash"])
print(show.to_string())

In [ ]:
print("Run epochs per subject:", runs.groupby("subject").size().loc[rt.SUBJECTS].to_dict())
print("Epoch length (min): median", runs.minutes.median(), "range", runs.minutes.min(), "to", runs.minutes.max(),
      "(Methods: 45 to 90 min)")
print("forageNum values (forage assist when above 1):", runs.forageNum.fillna("not in log header").value_counts().to_dict())
print("outerReps settings:", runs.outerReps_setting.fillna("not in log header").value_counts().to_dict())
print("\nValid (non-timeout) trials per subject:", runs.groupby("subject").valid.sum().loc[rt.SUBJECTS].to_dict())
print("Timeout trials per subject:", runs.groupby("subject").lockout.sum().loc[rt.SUBJECTS].to_dict())
print("\nLogged rewards equal 'outer reward' lines in", int((runs.rewards_log == runs.reward_lines).sum()), "of", len(runs), "epochs")
print("Logged rewards equal DIO pump onsets in", int((runs.rewards_log == runs.pumps_dio).sum()), "of", len(runs), "epochs")
print(runs.loc[runs.rewards_log != runs.pumps_dio, ["subject", "day", "epoch", "rewards_log", "pumps_dio", "unmatched_pumps"]].to_string())
# Rewarded only when the visited arm is the goal the log printed for that trial
allv = pd.concat([r["trials"][~r["trials"].lockout] for r in runs.run])
print("\nRewarded exactly when the visited arm is the printed goal:", bool((allv.rewarded == (allv.outer == allv.goal)).all()),
      f"({len(allv)} trials)")
# Rewards per completed block against the programmed number
comp = []
for _, row in runs.iterrows():
    lab = rt.label_task_phase(row.run["trials"])
    for k in lab.block.unique():
        b = lab[lab.block == k]
        if b.final.any():
            comp.append({"subject": row.subject, "rewards": int(b.rewarded[~b.final].sum())})
comp = pd.DataFrame(comp)
print("\nRewards per completed block (Methods: 15, then 10, then 4 to 12 for two subjects):")
print(pd.crosstab(comp.subject, comp.rewards).loc[rt.SUBJECTS].to_string())

Trials are rebuilt from the StateScript log (see `replaytask.py`). In every one of the 116 run
epochs, the logged rewards equal the log's own `outer reward` lines, and a trial is rewarded
exactly when the visited arm is the goal printed for that trial. The logged rewards equal the
reward pump onsets of the DIO channels in 115 epochs. In jaq day 6 epoch 4 the DIO has one more
pump (arm 8) 226 s after the last logged reward, with no log line, so the log appears to stop
before the epoch ends. The number of rewards per completed block takes only the programmed values
(15 and 10, and 4 to 12 for despereaux and montague), which agrees with the Methods and
Supplementary Figure 1A. No epoch sets forage assist (forageNum is 1 in the 111 logs whose
header holds it; five logs carry no Python header).

In [ ]:
print("StateScript clock to NWB clock, fit on rewarded entries against DIO pump onsets:")
print("slope range:", runs.clock_slope.min(), "to", runs.clock_slope.max())
print("median residual per epoch (ms): median", runs.clock_median_ms.median(), "max", runs.clock_median_ms.max())
print("largest residual per epoch (ms): median", runs.clock_max_ms.median(), "; epochs above 50 ms:")
print(runs.loc[runs.clock_max_ms > 50, ["subject", "day", "epoch", "clock_median_ms", "clock_max_ms"]].to_string())
bad = runs.loc[runs.clock_max_ms > 50].iloc[0].run
tr = bad["trials"]
rt_s = tr[tr.rewarded & ~tr.lockout].t_ms.to_numpy() / 1000
print("Interval differences between consecutive rewards, log minus DIO (s), in that epoch:",
      np.round(np.diff(rt_s) - np.diff(bad["dio_pump_t"]), 2)[:8], "...")

# Position against DIO beam breaks: decode the poked arm from position at the beam break
acc = []
for _, row in runs.iterrows():
    a0, n = rt.arm_from_position(row.run, 0.0)
    a1, _ = rt.arm_from_position(row.run, -20.0)
    a2, _ = rt.arm_from_position(row.run, 20.0)
    acc.append({"subject": row.subject, "acc_0s": a0, "acc_minus20s": a1, "acc_plus20s": a2, "entries": n})
acc = pd.DataFrame(acc)
print("\nPoked arm decoded from head position at the DIO beam break (leave-one-out nearest centroid; chance about 0.125):")
print(acc.groupby("subject")[["acc_0s", "acc_minus20s", "acc_plus20s"]].median().loc[rt.SUBJECTS].round(3).to_string())
print("epochs:", len(acc), "; outer beam breaks per epoch: median", acc.entries.median(),
      "; lowest accuracy at 0 s:", round(acc.acc_0s.min(), 3))

The StateScript times map onto the NWB clock with a slope of 1 and a median residual of a few
milliseconds, so the trial table and the DIO events share one clock. In montague day 16 epoch 2
the log's clock steps by about 3.4 s twice against the DIO early in the epoch; the trials are
unaffected because the analysis uses their order, not their times. Head position at the moment
of each outer-port beam break identifies the arm, while position taken 20 s earlier or later
does not, so position and the DIO share a clock. The raw broadband is not used here; its
timestamps are checked only for the span below.

In [ ]:
pos = runs[["subject", "pos_fs", "pos_nan_t", "pos_nonincreasing", "pos_unit", "pos_x_range_m", "pos_y_range_m"]]
print("Position: unit", pos.pos_unit.unique().tolist(), "; sampling rate (Hz)", pos.pos_fs.min(), "to", pos.pos_fs.max(),
      "(Methods: 30 Hz)")
print("x range (m):", pos.pos_x_range_m.min(), "to", pos.pos_x_range_m.max(), "; y range (m):",
      pos.pos_y_range_m.min(), "to", pos.pos_y_range_m.max())
print("Epochs with NaN position timestamps:", int((pos.pos_nan_t > 0).sum()), "of", len(pos), "; NaN timestamps per epoch up to",
      int(pos.pos_nan_t.max()))
print("Epochs with non-increasing position timestamps:", int((pos.pos_nonincreasing > 0).sum()), "; most in one epoch:",
      int(pos.pos_nonincreasing.max()))
print("\nRun epochs whose StateScript log is identical to another's:", int(runs.sc_hash.duplicated().sum()))
ephys = pd.DataFrame([{"subject": d["subject"], "day": d["day"], "samples": d["n_samples"],
                       "span_s": d["ephys_t1"] - d["ephys_t0"], "chunks": str(d["ephys_chunks"]),
                       "conversion": d["ephys_conversion"]} for d in days])
ephys["rate_over_span"] = ephys.samples / ephys.span_s
print("Raw broadband: chunk shapes", ephys.chunks.unique().tolist(), "; conversion to volts", ephys.conversion.unique().tolist())
print("samples / time span (Hz): median", round(ephys.rate_over_span.median()), "range",
      round(ephys.rate_over_span.min()), "to", round(ephys.rate_over_span.max()),
      "(30 kHz with the gaps between epochs inside the span)")
ephys["epoch_seconds"] = [d["epoch_seconds"] for d in days]
ephys["samples_over_epoch_time"] = ephys.samples / ephys.epoch_seconds
print("samples / summed epoch durations (Hz): median", round(ephys.samples_over_epoch_time.median()), "range",
      round(ephys.samples_over_epoch_time.min()), "to", round(ephys.samples_over_epoch_time.max()))
print("Recorded hours of raw broadband at 30 kHz:", round(ephys.samples.sum() / 30000 / 3600, 1))

# Pinned version against the draft, for one file
cmp = rt.compare_with_draft("sub-montague/sub-montague_ses-montague-04_behavior+ecephys.nwb")
pub, dr = cmp[rt.DANDISET_VERSION], cmp["draft"]
print("\nmontague day 4, pinned version against draft:")
for v in (rt.DANDISET_VERSION, "draft"):
    print(f"  {v}: NWB {cmp[v]['nwb_version']}, file created {cmp[v]['file_create_date'][:10]}, "
          f"position timestamps {np.nanmin(cmp[v]['pos_t']):.1f} to {np.nanmax(cmp[v]['pos_t']):.1f} s, "
          f"home beam timestamps {cmp[v]['dio_t'].min():.1f} to {cmp[v]['dio_t'].max():.1f} s")
print("  largest home beam timestamp difference (s):", float(np.max(np.abs(pub['dio_t'] - dr['dio_t']))))
print("  position timestamp difference, draft minus pinned (s): median",
      float(np.nanmedian(dr['pos_t'] - pub['pos_t'])), "; position values identical:", cmp["pos_data_equal"])

Position is stated in metres and spans about 1.1 to 1.3 m on each axis, which fits an
eight-arm maze; it is sampled at about 33 Hz, not the 30 Hz of the Methods. 110 of 116 epochs end
with 1 to 4 NaN timestamps, and two have non-increasing ones; these samples are dropped here. No
two run epochs share a StateScript log, so no recording was deposited twice. The raw broadband
has 30,000 samples per second of summed epoch time, as the Methods state. In the draft, the
position timestamps of the file compared are shifted by about -1.59e9 s from the DIO clock, with
the same position values; the pinned version is used throughout.

Sessions 1 and 3 of subject 1. The Methods exclude "sessions 1 and 3 of subject 1"
for decoding problems; the second was a session where "not all arms were visited". The files do
not mark these sessions. In order of recording, jaq's run sessions 1 and 3 are day 6 epochs 2
and 5, and all eight arms are visited in both. The only jaq session with fewer than eight arms
visited is day 6 epoch 4 (session 2 in order). The readings run for fig1d and fig1ef are
therefore (a) sessions 1 and 3 in order of recording and (b) day 6 epochs 2 and 4; with no
exclusion (c) is shown as a variant, not a reading, since the criterion excludes two sessions (a
question has been asked).

In [ ]:
jaq = runs[runs.subject == "jaq"][["session", "day", "epoch", "arms_visited", "valid", "rewards_log"]]
print(jaq.head(6).to_string(index=False))
EXCLUDE = {"a: sessions 1 and 3 in order": [(6, 2), (6, 5)],
           "b: day 6 epochs 2 and 4": [(6, 2), (6, 4)],
           "c, variant: none": []}

def session_set(reading):
    ex = EXCLUDE[reading]
    keep = ~((runs.subject == "jaq") & runs.apply(lambda r: (r.day, r.epoch) in ex, axis=1))
    return runs[keep]

# Streaming cost of the raw broadband, measured
url = assets.url[assets.path.str.contains("jaq-10")].iloc[0]
secs, nbytes = rt.time_raw_read(url)
rate = nbytes / secs
total = files.GB.sum() * 1e9
print(f"\nRead {nbytes / 1e6:.0f} MB of raw broadband (300000 samples x 128 channels) in {secs:.1f} s, {rate / 1e6:.1f} MB/s")
print(f"At that rate the dandiset's {total / 1e12:.1f} TB would take {total / rate / 3600:.0f} h; "
      f"one 90-min run epoch ({90 * 60 * 30000 * 128 * 2 / 1e9:.0f} GB) {90 * 60 * 30000 * 128 * 2 / rate / 60:.0f} min")

The raw broadband is stored one channel per chunk of 212,978 to 329,259 samples (7 to 11 s),
so any time window must fetch every channel's chunk. The rate measured above puts streaming the
run epochs alone (116 epochs of about 41 GB) far beyond the registry's 30-minute limit, before
any filtering, spike detection or decoding. This is why the 31 replay and ripple claims are
blocked as too_large.

## fig1d-sessions-blocks

> The analysis includes 24, 32, 23 and 33 behavioural sessions and 89, 85, 105 and 166 trial
> blocks for the four subjects.

Criterion: sessions are the full-task sessions less sessions 1 and 3 of subject 1; blocks are
those with a trial of task phase >= 4 (`any(ctphase>=4)`, dfs_ripcontent.m part 2). Reproduced
when all eight counts are within tolerance. Each session reading (above) and each reading of the
block's final trial (`replaytask.FINAL_TRIAL_READINGS`) is run.

In [ ]:
EXPECTED_SESSIONS = {"jaq": 24, "roquefort": 32, "despereaux": 23, "montague": 33}
EXPECTED_BLOCKS = {"jaq": 89, "roquefort": 85, "despereaux": 105, "montague": 166}
rows, per_session = [], []
for reading in EXCLUDE:
    ss = session_set(reading)
    for final in rt.FINAL_TRIAL_READINGS:
        for s in rt.SUBJECTS:
            sub = ss[ss.subject == s]
            nb = [len(rt.full_blocks(rt.label_task_phase(r["trials"], final))) for r in sub.run]
            rows.append({"sessions_reading": reading, "final_trial": final, "subject": s,
                         "sessions": len(sub), "expected_sessions": EXPECTED_SESSIONS[s],
                         "blocks": sum(nb), "expected_blocks": EXPECTED_BLOCKS[s],
                         "blocks_per_session_median": np.median(nb), "blocks_per_session_range": f"{min(nb)}-{max(nb)}"})
res1d = pd.DataFrame(rows)
res1d["sessions_ok"] = (res1d.sessions - res1d.expected_sessions).abs() <= 0.1 * res1d.expected_sessions
res1d["blocks_ok"] = (res1d.blocks - res1d.expected_blocks).abs() <= 0.1 * res1d.expected_blocks
print(res1d.to_string(index=False))
print("\nAll eight counts within tolerance, per reading:")
print(res1d.groupby(["sessions_reading", "final_trial"]).apply(lambda g: bool(g.sessions_ok.all() and g.blocks_ok.all())).to_string())

**Result: reproduced.** Under both readings of the excluded sessions and both readings of the
final trial, all eight counts are within 10%. Roquefort and montague match the paper's session
counts exactly (32 and 33), roquefort its block count (85), and montague has 166 or 167 blocks
against 166, depending on the final-trial reading. Jaq has 25 sessions against 24 and 88 (reading
a) or 91 (reading b) blocks against 89. Despereaux has 24 sessions against 23 and 109 blocks
against 105. With no session excluded (variant c), jaq has 27 sessions, outside the tolerance of
24, and 96 blocks, inside that of 89. One run epoch each of jaq and despereaux is in the files but
not in the paper's count.

## fig1ef-reward-rate

> Aligned to the goal change, the outer-arm reward rate starts at chance (0.125) and rises;
> aligned to the first reward at the new goal, it is well above chance, so all subjects keep
> visiting a goal once found.

Criterion: per subject over the blocks with at least four rewards, the final unrewarded trial
dropped; a trial scores 1 when rewarded; binomial S.E.M.; trials 1 to 15. Reproduced when, in all
four subjects, the band on trial 1 of Figure 1E reaches 0.125 or lies below it, the rate rises
over trials 1 to 15, and every point of Figure 1F has its band above 0.125. "Rises" is shown two
ways: trial 15 against trial 1 (non-overlapping bands), and the Spearman correlation of the
rate with trial number.

In [ ]:
def curves(reading, final):
    ss = session_set(reading)
    out = {}
    for s in rt.SUBJECTS:
        S, R = [], []
        for r in ss[ss.subject == s].run:
            a, b = rt.reward_curves(rt.label_task_phase(r["trials"], final))
            S.append(a); R.append(b)
        out[s] = (np.vstack(S), np.vstack(R))
    return out

rows = []
for reading in EXCLUDE:
    for final in rt.FINAL_TRIAL_READINGS:
        cv = curves(reading, final)
        for s in rt.SUBJECTS:
            p, se, n = rt.binomial_curve(cv[s][0])
            p2, se2, n2 = rt.binomial_curve(cv[s][1])
            rho, prho = stats.spearmanr(rt.CURVE_TRIALS, p)
            rows.append({"sessions_reading": reading, "final_trial": final, "subject": s, "blocks": len(cv[s][0]),
                         "E_trial1": round(p[0], 3), "E_trial1_lower": round(p[0] - se[0], 3),
                         "E_trial15": round(p[14], 3), "E_trial15_lower": round(p[14] - se[14], 3),
                         "E_rho": round(rho, 3), "E_rho_p": prho,
                         "F_min": round(p2.min(), 3), "F_min_lower": round((p2 - se2).min(), 3),
                         "F_n_trial15": int(n2[14])})
res1ef = pd.DataFrame(rows)
res1ef["E_starts_at_chance"] = res1ef.E_trial1_lower <= rt.CHANCE
res1ef["E_rises"] = (res1ef.E_trial15_lower > res1ef.E_trial1 + (res1ef.E_trial1 - res1ef.E_trial1_lower)) & (res1ef.E_rho > 0) & (res1ef.E_rho_p < 0.05)
res1ef["F_above_chance"] = res1ef.F_min_lower > rt.CHANCE
print(res1ef.to_string(index=False))
print("\nAll three conditions in all four subjects, per reading:")
print(res1ef.groupby(["sessions_reading", "final_trial"]).apply(
    lambda g: bool(g.E_starts_at_chance.all() and g.E_rises.all() and g.F_above_chance.all())).to_string())

In [ ]:
cv = curves("a: sessions 1 and 3 in order", "old_goal_visit")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
cols = ["#1b5c29", "#197b64", "#219fa9", "#7be1bf"]
for s, c in zip(rt.SUBJECTS, cols):
    for ax, mat in zip(axes, cv[s]):
        p, se, n = rt.binomial_curve(mat)
        ax.fill_between(rt.CURVE_TRIALS, p - se, p + se, color=c, alpha=0.3, lw=0)
        ax.plot(rt.CURVE_TRIALS, p, ".-", color=c, label=f"{s} ({len(mat)} blocks)")
for ax, t in zip(axes, ["trials since goal change (1E)", "trials since first reward at new goal (1F)"]):
    ax.axhline(rt.CHANCE, color="k", ls=":")
    ax.set_xlabel(t)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("reward rate")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
lo = min(np.nanmin(rt.binomial_curve(m)[0] - rt.binomial_curve(m)[1]) for s in rt.SUBJECTS for m in cv[s])
hi = max(np.nanmax(rt.binomial_curve(m)[0] + rt.binomial_curve(m)[1]) for s in rt.SUBJECTS for m in cv[s])
print(f"Plotted values (mean +- S.E.M.) span {lo:.3f} to {hi:.3f}; none fall outside the axes (0 to 1).")

**Result: reproduced.** Under every reading, in all four subjects, the band on the first trial
after the goal change reaches 0.125 or lies below it (trial 1 means 0.068 to 0.133), the rate rises
to 0.58 to 0.70 by trial 15 with a positive Spearman correlation, and after the first reward at
the new goal every point of the band is above 0.125 (lowest lower bound 0.234, roquefort).

## Claims not tested

The 31 other claims rest on sharp-wave ripples, multiunit events or the clusterless state space
decoder. The dandiset holds neither these nor the spikes or LFP they are computed from, only the
raw 30 kHz broadband, about 9.1 TB in total and stored one channel per chunk. The data check above
measures the streaming rate and shows that reading the run epochs once would take far longer than
the 30-minute limit, before any spike detection, ripple filtering or decoding. They are blocked
as too_large. Ripple detection would also need the CA1 cell-layer tetrodes and the corpus
callosum reference tetrode, which the electrode table does not identify (every location is
"None"). fig6b-visits-decay is behavioural in part, but its criterion compares the visit decay
with the replay enrichment of fig6a-pg-replay-persists, so it is blocked with it.

## Summary

| claim | outcome |
|---|---|
| fig1d-sessions-blocks | reproduced |
| fig1ef-reward-rate | reproduced |
| the 31 replay, ripple and decoding claims | blocked, too_large |

How the analysis differs from the paper's. Trials are rebuilt from the StateScript logs in the
files, not from the paper's preprocessed trial structures. Task phases follow the claims file's
account of label_trial_interval.m. Two details it leaves open are run both ways: which jaq
sessions are "sessions 1 and 3", and which trial closes a block when the rat does not go back
to the old goal right after the switch. All readings give the same outcomes.

What was found about the dataset.
- The files hold raw broadband, behavioural events, position and task logs, but no spikes, LFP,
  ripple times, linearised position or decoding results. Every replay finding of the paper has to
  be recomputed from about 9.1 TB of raw data, which this registry cannot do.
- The electrode table gives no location for any of the 128 channels, so the CA1 tetrodes and the
  reference tetrode cannot be identified.
- The files do not mark which run sessions the paper analysed. They hold 27 and 24 run epochs
  for jaq and despereaux, where the paper used 26 (before excluding two) and 23.
- The behaviour in the logs matches the hardware: rewards equal the pump onsets in 115 of 116
  epochs. Under the readings of the claim, the block counts are within 4% of the paper's for
  jaq, despereaux and montague, and equal for roquefort.
- The draft version's files were rewritten in 2022, and in the file compared its position
  timestamps are off the DIO clock by about -1.59e9 s; the published version is consistent.
- Position is sampled at about 33 Hz (Methods: 30 Hz); most epochs end with NaN timestamps.
- In one epoch (montague day 16 epoch 2) the log's clock steps by about 3.4 s against the DIO,
  and in one (jaq day 6 epoch 4) the log stops before a final pump.